In [ ]:
from sklearn.mixture import GaussianMixture
from app.preprocessing.preprocessing import preprocessing , PCA_transform
from models.__pycache__.gmm_model import train_gmm , predict_gmm, GMM_CONFIGS
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import pairwise_distances

In [8]:
df = pd.read_csv("../cross_genre_music_discovery/raw_data/spotify-tracks-dataset-detailed.csv")

X_scaled, df_metadata, scaler, features = preprocessing(df)

In [25]:
df[df["track_id"] == "25hbD2ZhTlBv0MQAAAI6DY"]

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
5624,25hbD2ZhTlBv0MQAAAI6DY,FictionJunction;LiSA,from the edge -TV ver.-,from the edge -TV ver.-,47,90102,False,0.416,0.922,8,-4.044,1,0.162,0.238,0.0,0.059,0.161,162.167,4,anime


In [9]:
df_scaled = pd.DataFrame(X_scaled, columns=features)

In [10]:
df_completo = pd.concat([df_metadata, df_scaled], axis=1)
df_completo.head()

,track_id,artists,track_name,genres,danceability,energy,loudness,speechiness,acousticness,instrumentalness,valence,tempo
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,"[acoustic, j-pop, singer-songwriter, songwriter]",0.676937,-0.701560,0.323475,0.776805,-0.854487,-0.551196,0.960811,-1.182342
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost - Acoustic,"[acoustic, chill]",-0.815682,-1.857298,-1.792992,-0.041318,1.801679,-0.551182,-0.762295,-1.535282
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,[acoustic],-0.710732,-1.101171,-0.279443,-0.293991,-0.324922,-0.551199,-1.327690,-1.574441
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Can't Help Falling In Love,[acoustic],-1.713586,-2.274147,-2.051270,-0.531946,1.745089,-0.550983,-1.239227,1.993142
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,[acoustic],0.338766,-0.772080,-0.268748,-0.332015,0.446492,-0.551199,-1.146917,-0.098203


In [11]:
df_completo.shape

(80175, 12)

# Verificando os clusters

In [12]:
# Selecionando de forma aleatória as tracks de teste

tracks_teste = (
    df_completo["track_id"]
    .dropna()
    .drop_duplicates()
    .sample(10, random_state=1)
    .tolist()
)

print(tracks_teste)
print("Quantidade:", len(tracks_teste))

['2jLxGO8FukajZNDvE0rmf4', '73lIldE4SbKxglgJXiQ7cd', '0ZSHT8qBpxOIGoKxEslL4V', '6nzXkCBOhb2mxctNihOqbb', '0pg2VKzTsBdbxPzcOkReY2', '0LIEeAQBYYf4go8DEL5R5H', '6IaSUc4wx1rzo8lGs4KOot', '3j0oWeUZjTA0HDUS5j64Vm', '1uvBOCykry1KtIC3Rh7kMe', '25hbD2ZhTlBv0MQAAAI6DY']
Quantidade: 10


In [13]:
input_df = df_completo[
    df_completo["track_id"].isin(tracks_teste)
].copy()

print("Formato:", input_df.shape)
print("IDs únicos:", input_df["track_id"].nunique())

Formato: (10, 12)
IDs únicos: 10


In [17]:
input_df

,track_id,artists,track_name,genres,danceability,energy,loudness,speechiness,acousticness,instrumentalness,valence,tempo
4567,25hbD2ZhTlBv0MQAAAI6DY,FictionJunction;LiSA,from the edge -TV ver.-,[anime],-0.839004,1.104524,0.868684,1.009853,-0.241526,-0.551199,-1.169995,1.330685
12076,2jLxGO8FukajZNDvE0rmf4,Aaron Smith;Luvli;Philip George,Dancin (feat. Luvli) - Philip George Remix,[chicago-house],0.881006,-0.348963,-0.604510,-0.345507,-0.942947,-0.548481,0.649267,0.038938
12522,6IaSUc4wx1rzo8lGs4KOot,Hisohkah;WMD,School Rooftop - Slowed Down Version,"[chill, sad]",1.050092,-1.508618,0.213101,-0.322202,-0.131324,2.179556,-0.900759,0.667347
16071,1uvBOCykry1KtIC3Rh7kMe,Alesso;TINI,Sad Song,"[dance, edm, electro, house]",1.020940,0.199523,0.378157,-0.533172,-0.689482,-0.551199,0.318492,-0.434323
36629,0LIEeAQBYYf4go8DEL5R5H,Nosferatu;Spitnoise,Braindead Lunatic,[happy],-1.387076,1.257316,1.539601,2.812911,-0.866997,-0.551199,-1.077686,2.781369
37110,73lIldE4SbKxglgJXiQ7cd,Matanza,Maldito Hippie Sujo,"[hard-rock, punk, punk-rock]",-1.060565,1.374849,1.100125,-0.021692,-0.934726,-0.551199,1.110814,1.425520
43684,6nzXkCBOhb2mxctNihOqbb,M.I.A.,Bad Girls,"[indie, indie-pop]",0.461207,1.065346,1.155816,-0.222850,-0.947206,-0.551199,-0.469983,0.697131
45239,0ZSHT8qBpxOIGoKxEslL4V,Alphaxone,Mysterious,[iranian],-2.162538,-1.571302,-2.933854,-0.285405,1.947622,2.442246,-1.481539,0.293118
77395,0pg2VKzTsBdbxPzcOkReY2,Kid Loco,A grand love theme,[trip-hop],-0.040220,0.074155,-0.188037,-0.225303,0.544780,2.072647,-0.019975,2.063372
77676,3j0oWeUZjTA0HDUS5j64Vm,Gramatik,While I Was Playin' Fair,[trip-hop],1.131720,0.450259,0.667509,2.739316,-0.904525,-0.298589,-0.104592,-0.876920


In [14]:
modelos = {}

for nome, cfg in GMM_CONFIGS.items():

    n_pca = 3 if nome.startswith("pca3") else 5

    # Transformar os dados e guardar o PCA ajustado
    df_pca, pca_ajustado = PCA_transform(
        df_scaled,
        number_of_components=n_pca
    )

    modelos[nome] = {
        "model": train_gmm(df_pca, **cfg),
        "pca": pca_ajustado,
        "scaler": scaler
    }

In [15]:
modelos

{'pca3_tied15': {'model': GaussianMixture(covariance_type='tied', n_components=15, n_init=5,
                  random_state=42),
  'pca': PCA(n_components=3),
  'scaler': StandardScaler()},
 'pca5_diag30': {'model': GaussianMixture(covariance_type='diag', n_components=30, n_init=5,
                  random_state=42),
  'pca': PCA(n_components=5),
  'scaler': StandardScaler()}}

In [16]:
def atribuir_clusters(
    df_musicas,
    config,
    features,
    cluster_col="cluster_gmm"
):
    """
    Atribui clusters a músicas cujas features
    já estão escaladas.

    Retorna:
    - DataFrame com a coluna de clusters
    - Coordenadas no espaço usado pelo modelo
    """

    # 1. Recuperar as features já escaladas
    X_scaled = df_musicas[features].to_numpy()

    # 2. Aplicar o PCA treinado, quando existir
    if config.get("pca") is not None:
        X_modelo = config["pca"].transform(X_scaled)
    else:
        X_modelo = X_scaled

    # 3. Identificar o cluster de cada música
    clusters = config["model"].predict(X_modelo)

    # 4. Adicionar os clusters ao DataFrame
    resultado = df_musicas.copy()
    resultado[cluster_col] = clusters

    # 5. Retornar os dados e as coordenadas
    return resultado, X_modelo


In [17]:
df_with_cluster, coordenates = atribuir_clusters(
    df_musicas= input_df,
    config = modelos["pca3_tied15"],
    features = features,
    cluster_col="cluster_gmm"
    )

/home/victor/.pyenv/versions/3.10.6/envs/cross_genre_music_discovery/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/home/victor/.pyenv/versions/3.10.6/envs/cross_genre_music_discovery/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(


In [18]:
df_with_cluster

,track_id,artists,track_name,genres,danceability,energy,loudness,speechiness,acousticness,instrumentalness,valence,tempo,cluster_gmm
4567,25hbD2ZhTlBv0MQAAAI6DY,FictionJunction;LiSA,from the edge -TV ver.-,[anime],-0.839004,1.104524,0.868684,1.009853,-0.241526,-0.551199,-1.169995,1.330685,6
12076,2jLxGO8FukajZNDvE0rmf4,Aaron Smith;Luvli;Philip George,Dancin (feat. Luvli) - Philip George Remix,[chicago-house],0.881006,-0.348963,-0.604510,-0.345507,-0.942947,-0.548481,0.649267,0.038938,9
12522,6IaSUc4wx1rzo8lGs4KOot,Hisohkah;WMD,School Rooftop - Slowed Down Version,"[chill, sad]",1.050092,-1.508618,0.213101,-0.322202,-0.131324,2.179556,-0.900759,0.667347,10
16071,1uvBOCykry1KtIC3Rh7kMe,Alesso;TINI,Sad Song,"[dance, edm, electro, house]",1.020940,0.199523,0.378157,-0.533172,-0.689482,-0.551199,0.318492,-0.434323,9
36629,0LIEeAQBYYf4go8DEL5R5H,Nosferatu;Spitnoise,Braindead Lunatic,[happy],-1.387076,1.257316,1.539601,2.812911,-0.866997,-0.551199,-1.077686,2.781369,12
37110,73lIldE4SbKxglgJXiQ7cd,Matanza,Maldito Hippie Sujo,"[hard-rock, punk, punk-rock]",-1.060565,1.374849,1.100125,-0.021692,-0.934726,-0.551199,1.110814,1.425520,13
43684,6nzXkCBOhb2mxctNihOqbb,M.I.A.,Bad Girls,"[indie, indie-pop]",0.461207,1.065346,1.155816,-0.222850,-0.947206,-0.551199,-0.469983,0.697131,13
45239,0ZSHT8qBpxOIGoKxEslL4V,Alphaxone,Mysterious,[iranian],-2.162538,-1.571302,-2.933854,-0.285405,1.947622,2.442246,-1.481539,0.293118,7
77395,0pg2VKzTsBdbxPzcOkReY2,Kid Loco,A grand love theme,[trip-hop],-0.040220,0.074155,-0.188037,-0.225303,0.544780,2.072647,-0.019975,2.063372,6
77676,3j0oWeUZjTA0HDUS5j64Vm,Gramatik,While I Was Playin' Fair,[trip-hop],1.131720,0.450259,0.667509,2.739316,-0.904525,-0.298589,-0.104592,-0.876920,1


In [19]:
coordenates

array([[ 1.18336269, -1.71562726,  0.60043453],
       [ 0.40646266,  0.85388595, -0.2398906 ],
       [-1.11458243, -0.5440575 ,  0.73647854],
       [ 0.90370842,  0.73963966, -0.57041191],
       [ 2.36347147, -2.61486323,  2.11265161],
       [ 2.19203349, -0.95286455, -0.61582585],
       [ 1.79822078, -0.70324909, -0.40322583],
       [-4.73691626, -1.69841987,  0.72978456],
       [-0.4919952 , -1.19072241,  0.60507851],
       [ 1.55359412,  0.61140577,  2.44804228]])

In [20]:
modelos

{'pca3_tied15': {'model': GaussianMixture(covariance_type='tied', n_components=15, n_init=5,
                  random_state=42),
  'pca': PCA(n_components=3),
  'scaler': StandardScaler()},
 'pca5_diag30': {'model': GaussianMixture(covariance_type='diag', n_components=30, n_init=5,
                  random_state=42),
  'pca': PCA(n_components=5),
  'scaler': StandardScaler()}}

In [21]:
print(modelos.keys())

dict_keys(['pca3_tied15', 'pca5_diag30'])


In [22]:
input_df

,track_id,artists,track_name,genres,danceability,energy,loudness,speechiness,acousticness,instrumentalness,valence,tempo
4567,25hbD2ZhTlBv0MQAAAI6DY,FictionJunction;LiSA,from the edge -TV ver.-,[anime],-0.839004,1.104524,0.868684,1.009853,-0.241526,-0.551199,-1.169995,1.330685
12076,2jLxGO8FukajZNDvE0rmf4,Aaron Smith;Luvli;Philip George,Dancin (feat. Luvli) - Philip George Remix,[chicago-house],0.881006,-0.348963,-0.604510,-0.345507,-0.942947,-0.548481,0.649267,0.038938
12522,6IaSUc4wx1rzo8lGs4KOot,Hisohkah;WMD,School Rooftop - Slowed Down Version,"[chill, sad]",1.050092,-1.508618,0.213101,-0.322202,-0.131324,2.179556,-0.900759,0.667347
16071,1uvBOCykry1KtIC3Rh7kMe,Alesso;TINI,Sad Song,"[dance, edm, electro, house]",1.020940,0.199523,0.378157,-0.533172,-0.689482,-0.551199,0.318492,-0.434323
36629,0LIEeAQBYYf4go8DEL5R5H,Nosferatu;Spitnoise,Braindead Lunatic,[happy],-1.387076,1.257316,1.539601,2.812911,-0.866997,-0.551199,-1.077686,2.781369
37110,73lIldE4SbKxglgJXiQ7cd,Matanza,Maldito Hippie Sujo,"[hard-rock, punk, punk-rock]",-1.060565,1.374849,1.100125,-0.021692,-0.934726,-0.551199,1.110814,1.425520
43684,6nzXkCBOhb2mxctNihOqbb,M.I.A.,Bad Girls,"[indie, indie-pop]",0.461207,1.065346,1.155816,-0.222850,-0.947206,-0.551199,-0.469983,0.697131
45239,0ZSHT8qBpxOIGoKxEslL4V,Alphaxone,Mysterious,[iranian],-2.162538,-1.571302,-2.933854,-0.285405,1.947622,2.442246,-1.481539,0.293118
77395,0pg2VKzTsBdbxPzcOkReY2,Kid Loco,A grand love theme,[trip-hop],-0.040220,0.074155,-0.188037,-0.225303,0.544780,2.072647,-0.019975,2.063372
77676,3j0oWeUZjTA0HDUS5j64Vm,Gramatik,While I Was Playin' Fair,[trip-hop],1.131720,0.450259,0.667509,2.739316,-0.904525,-0.298589,-0.104592,-0.876920


In [33]:
input_df["cluster_gmm"] = clusters

input_df[
    ["track_name", "artists", "genres", "cluster_gmm"]
]

,track_name,artists,genres,cluster_gmm
4567,from the edge -TV ver.-,FictionJunction;LiSA,[anime],6
12076,Dancin (feat. Luvli) - Philip George Remix,Aaron Smith;Luvli;Philip George,[chicago-house],9
12522,School Rooftop - Slowed Down Version,Hisohkah;WMD,"[chill, sad]",10
16071,Sad Song,Alesso;TINI,"[dance, edm, electro, house]",9
36629,Braindead Lunatic,Nosferatu;Spitnoise,[happy],12
37110,Maldito Hippie Sujo,Matanza,"[hard-rock, punk, punk-rock]",13
43684,Bad Girls,M.I.A.,"[indie, indie-pop]",13
45239,Mysterious,Alphaxone,[iranian],7
77395,A grand love theme,Kid Loco,[trip-hop],6
77676,While I Was Playin' Fair,Gramatik,[trip-hop],1


In [91]:
print("Músicas no input:", len(input_df))
print("Features utilizadas:", X_input.shape[1])
print("Componentes do PCA:", config["pca"].n_components_)
print("Gaussianas do GMM:", config["model"].n_components)

Músicas no input: 10
Features utilizadas: 8
Componentes do PCA: 3
Gaussianas do GMM: 15


In [34]:
grupos = input_df.groupby("cluster_gmm")

for cluster, grupo in grupos:
    print(
        f"Cluster {cluster}: "
        f"{len(grupo)} música(s)"
    )

Cluster 1: 1 música(s)
Cluster 6: 2 música(s)
Cluster 7: 1 música(s)
Cluster 9: 2 música(s)
Cluster 10: 1 música(s)
Cluster 12: 1 música(s)
Cluster 13: 2 música(s)


In [35]:
grupo_0 = input_df[
    input_df["cluster_gmm"] == 0
]

In [36]:
df_coordenadas = pd.DataFrame(
    X_input_pca,
    index=input_df.index
)

In [37]:
coordenadas_0 = df_coordenadas.loc[grupo_0.index]

In [38]:
distancias_0 = pairwise_distances(
    coordenadas_0,
    metric="euclidean"
)

ValueError: Found array with 0 sample(s) (shape=(0, 3)) while a minimum of 1 is required by check_pairwise_arrays.

In [104]:
soma_distancias = distancias_0.sum(axis=1)

posicao_medoide = soma_distancias.argmin()

medoide = grupo_0.iloc[posicao_medoide]

In [105]:
print("Medoide:", medoide["track_name"])
print("Artista:", medoide["artists"])
print("Cluster:", medoide["cluster_gmm"])

Medoide: Chuttesai Chuttesai
Artista: Yuvan Shankar Raja;Karthik;Sunitha Sarathy
Cluster: 0


# Função para achar medoides

In [39]:
def find_medoids(input_df, X_input, cluster_col="cluster_gmm"):

    # Criar um DataFrame com as coordenadas das músicas
    df_coordenadas = pd.DataFrame(
        X_input,
        index=input_df.index
    )

    medoides = []

    # Percorrer os clusters presentes no input
    for cluster, grupo in input_df.groupby(cluster_col):

        # Selecionar as coordenadas das músicas do grupo
        coordenadas_grupo = df_coordenadas.loc[grupo.index]

        # Calcular as distâncias entre as músicas do grupo
        distancias = pairwise_distances(
            coordenadas_grupo,
            metric="euclidean"
        )

        # Somar as distâncias de cada música às demais
        soma_distancias = distancias.sum(axis=1)

        # Encontrar a posição da música mais central
        posicao_medoide = soma_distancias.argmin()

        # Recuperar o registro da música escolhida
        medoide = grupo.iloc[posicao_medoide].copy()

        # Guardar também o tamanho do grupo
        medoide["n_musicas"] = len(grupo)

        medoides.append(medoide)

    return pd.DataFrame(medoides).reset_index(drop=True)

## Teste da função

In [40]:
medoides_df = find_medoids(input_df, X_input, cluster_col="cluster_gmm")

In [41]:
print("Músicas de entrada:", len(input_df))

print("Clusters ocupados:", input_df["cluster_gmm"].nunique())

print("Medoides encontrados:", len(medoides_df))

print("Músicas representadas:", medoides_df["n_musicas"].sum())

Músicas de entrada: 10
Clusters ocupados: 7
Medoides encontrados: 7
Músicas representadas: 10


# Encontrar e excluir os gêneros presentes nas músicas imputadas

In [42]:
print(input_df["genres"].head(10).tolist())

[['anime'], ['chicago-house'], ['chill', 'sad'], ['dance', 'edm', 'electro', 'house'], ['happy'], ['hard-rock', 'punk', 'punk-rock'], ['indie', 'indie-pop'], ['iranian'], ['trip-hop'], ['trip-hop']]


In [43]:
print("Tipo do primeiro valor:")
print(type(input_df["genres"].iloc[0]))

Tipo do primeiro valor:
<class 'list'>


In [44]:
#encontra os gêneros das músicas do input e retorna um set

def get_input_genres(input_df):

    generos_input = set()

    for lista_generos in input_df["genres"]:
        generos_input.update(lista_generos)

    return generos_input

In [126]:
# encontra as músicas presentes nos clusters do input

def get_candidates(catalogo, medoides_df, cluster_col="cluster_gmm"):

    clusters = medoides_df[cluster_col].unique()

    candidatas = catalogo[
        catalogo[cluster_col].isin(clusters)
    ].copy()

    return candidatas

In [124]:
# Função que recebe as músicas dos clusters do input e considera apenas as com gêneros diferentes

def filtrar_generos(candidatas_df, generos_input):

    mascara = candidatas_df["genres"].apply(
        lambda lista: set(lista).isdisjoint(generos_input)
    )

    return candidatas_df[mascara].copy()

In [121]:
generos_input = set()

for lista_generos in input_df["genres"]:
    generos_input.update(lista_generos)

print("Quantidade de gêneros:", len(generos_input))
print("Gêneros excluídos:", sorted(generos_input))

Quantidade de gêneros: 12
Gêneros excluídos: ['brazil', 'children', 'dance', 'disco', 'electronic', 'funk', 'hard-rock', 'house', 'iranian', 'k-pop', 'party', 'swedish']


# Calculando a distância das músicas para o medoide 

In [127]:
def calcular_distancias(
    candidatas_df,
    medoides_df,
    X_candidatas,
    X_input,
    input_df,
    cluster_col="cluster_gmm"
):

    # Garantir que as coordenadas tenham os índices corretos
    coord_candidatas = pd.DataFrame(
        X_candidatas,
        index=candidatas_df.index
    )

    coord_input = pd.DataFrame(
        X_input,
        index=input_df.index
    )

    resultado = candidatas_df.copy()
    resultado["distancia_medoide"] = np.nan

    # Percorrer cada medoide
    for _, medoide in medoides_df.iterrows():

        cluster = medoide[cluster_col]

        # Selecionar as candidatas pertencentes ao mesmo cluster
        grupo_candidatas = resultado[
            resultado[cluster_col] == cluster
        ]

        if grupo_candidatas.empty:
            continue

        # Coordenadas das candidatas do cluster
        X_grupo = coord_candidatas.loc[grupo_candidatas.index]

        # Coordenadas do medoide
        X_medoide = coord_input.loc[
            input_df["track_id"] == medoide["track_id"]
        ]

        # Calcular a distância de cada candidata ao medoide
        distancias = pairwise_distances(
            X_grupo,
            X_medoide,
            metric="euclidean"
        ).ravel()

        # Armazenar as distâncias
        resultado.loc[
            grupo_candidatas.index,
            "distancia_medoide"
        ] = distancias

    return resultado

In [131]:
# ==========================================
# TESTE COMPLETO DO PIPELINE
# ==========================================

# 1. Selecionar o modelo treinado
config = modelos["pca3_tied15"]

# 2. Criar o catálogo com os clusters do GMM
# Transformar todas as músicas com scaler e PCA já treinados
X_catalogo_scaled = config["scaler"].transform(
    df_completo[features]
)

X_catalogo_pca = config["pca"].transform(
    X_catalogo_scaled
)

catalogo = df_completo.copy()
catalogo["cluster_gmm"] = config["model"].predict(
    X_catalogo_pca
)

print("=== CATÁLOGO ===")
print("Total de músicas:", len(catalogo))
print("Clusters:", catalogo["cluster_gmm"].nunique())

# 3. Preparar o input, usando as 10 músicas já selecionadas
input_df = df_completo[
    df_completo["track_id"].isin(input_df["track_id"])
].copy()

assert len(input_df) == 10
assert input_df["track_id"].nunique() == 10

# 4. Transformar as músicas do input
X_input_scaled = config["scaler"].transform(
    input_df[features]
)

X_input_pca = config["pca"].transform(
    X_input_scaled
)

# 5. Identificar os clusters do input
input_df["cluster_gmm"] = config["model"].predict(
    X_input_pca
)

# 6. Executar a função de medoides
medoides_df = find_medoids(
    input_df,
    X_input_pca
)

print("\n=== INPUT ===")
print("Músicas:", len(input_df))
print("Clusters ocupados:", input_df["cluster_gmm"].nunique())
print("Medoides:", len(medoides_df))

assert len(medoides_df) == input_df["cluster_gmm"].nunique()
assert medoides_df["n_musicas"].sum() == 10

# 7. Executar a função de gêneros proibidos
generos_input = get_input_genres(input_df)

print("\n=== GÊNEROS PROIBIDOS ===")
print("Quantidade:", len(generos_input))
print(sorted(generos_input))

# 8. Executar a função que recupera candidatas
candidatas_df = get_candidates(
    catalogo,
    medoides_df
)

print("\n=== CANDIDATAS ANTES DO FILTRO ===")
print("Quantidade:", len(candidatas_df))

# 9. Executar a função de filtragem por gênero
candidatas_filtradas = filtrar_generos(
    candidatas_df,
    generos_input
)

# Excluir as próprias músicas do usuário
candidatas_filtradas = candidatas_filtradas[
    ~candidatas_filtradas["track_id"].isin(input_df["track_id"])
].copy()

# Excluir IDs duplicados
candidatas_filtradas = candidatas_filtradas.drop_duplicates(
    subset="track_id"
)

print("\n=== CANDIDATAS APÓS FILTRAGEM ===")
print("Quantidade:", len(candidatas_filtradas))

# 10. Preparar coordenadas das candidatas
X_candidatas_scaled = config["scaler"].transform(
    candidatas_filtradas[features]
)

X_candidatas_pca = config["pca"].transform(
    X_candidatas_scaled
)

# 11. Executar a função de distância aos medoides
candidatas_com_distancias = calcular_distancias(
    candidatas_df=candidatas_filtradas,
    medoides_df=medoides_df,
    X_candidatas=X_candidatas_pca,
    X_input=X_input_pca,
    input_df=input_df
)

# ==========================================
# VALIDAÇÕES
# ==========================================

print("\n=== VALIDAÇÕES ===")

assert len(candidatas_com_distancias) > 0

assert candidatas_com_distancias[
    "distancia_medoide"
].notna().all()

assert (candidatas_com_distancias[
    "distancia_medoide"
] >= 0).all()

assert not candidatas_com_distancias[
    "track_id"
].isin(input_df["track_id"]).any()

assert candidatas_com_distancias["genres"].apply(
    lambda g: set(g).isdisjoint(generos_input)
).all()

assert candidatas_com_distancias["track_id"].is_unique

print("✅ Distâncias calculadas")
print("✅ Nenhum gênero proibido")
print("✅ Nenhuma música do input")
print("✅ Nenhum track_id duplicado")

print("\n=== CANDIDATAS POR CLUSTER ===")
display(
    candidatas_com_distancias.groupby("cluster_gmm")[
        "distancia_medoide"
    ].agg(["count", "min", "mean", "max"]).round(3)
)

print("\n=== EXEMPLO DE CANDIDATAS ===")
display(
    candidatas_com_distancias[
        [
            "track_name",
            "artists",
            "genres",
            "cluster_gmm",
            "distancia_medoide"
        ]
    ].sort_values("distancia_medoide").head(20)
)

print("\n✅ TESTE DE INTEGRAÇÃO CONCLUÍDO!")


/home/victor/.pyenv/versions/3.10.6/envs/cross_genre_music_discovery/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/home/victor/.pyenv/versions/3.10.6/envs/cross_genre_music_discovery/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(


=== CATÁLOGO ===
Total de músicas: 88751
Clusters: 15

=== INPUT ===
Músicas: 10
Clusters ocupados: 7
Medoides: 7

=== GÊNEROS PROIBIDOS ===
Quantidade: 12
['brazil', 'children', 'dance', 'disco', 'electronic', 'funk', 'hard-rock', 'house', 'iranian', 'k-pop', 'party', 'swedish']

=== CANDIDATAS ANTES DO FILTRO ===
Quantidade: 62372


/home/victor/.pyenv/versions/3.10.6/envs/cross_genre_music_discovery/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/home/victor/.pyenv/versions/3.10.6/envs/cross_genre_music_discovery/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(



=== CANDIDATAS APÓS FILTRAGEM ===
Quantidade: 53938

=== VALIDAÇÕES ===
✅ Distâncias calculadas
✅ Nenhum gênero proibido
✅ Nenhuma música do input
✅ Nenhum track_id duplicado

=== CANDIDATAS POR CLUSTER ===


/home/victor/.pyenv/versions/3.10.6/envs/cross_genre_music_discovery/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(


,count,min,mean,max
cluster_gmm,,,,
0,8740,0.205,4.065,9.147
4,3101,0.316,5.215,12.645
7,9377,0.191,4.112,10.550
10,15922,0.210,5.196,11.648
11,5398,0.147,3.959,7.944
13,1231,0.029,3.815,11.342
14,10169,0.421,15.779,90.851



=== EXEMPLO DE CANDIDATAS ===


,track_name,artists,genres,cluster_gmm,distancia_medoide
59810,Voices (They Want Me Dead),Convictions,[metalcore],13,0.028654
19112,Dance For Me (Eugy X Mr Eazi),Eugy;Mr Eazi,[dancehall],11,0.146845
59284,abcdefu,Our Last Night,[metalcore],7,0.190861
61252,A Estrada,Cidade Negra,[mpb],0,0.205171
21516,Satellite (feat. Arinn),Tobtok;Arinn,"[deep-house, groove]",0,0.208455
11516,借火,Juno Mak,[cantopop],10,0.210227
7903,You Do You,Zayde Wølf,[blues],0,0.216329
64038,Carícias - Ao Vivo,Art Popular,"[pagode, samba]",0,0.217229
48941,Blind Lead the Blind,Celldweller,[industrial],7,0.220294
42304,Feel It!,D-Block & S-te-Fan;D-Sturb,[hardstyle],7,0.223793



✅ TESTE DE INTEGRAÇÃO CONCLUÍDO!


# Função de recomendação

In [132]:
def recomendar_musicas(
    candidatas_df,
    medoides_df,
    modo="seguro",
    random_state=42
):
    # Gerador de números aleatórios reproduzíveis
    rng = np.random.default_rng(random_state)

    # Quantidade de músicas do input em cada cluster
    contagens = medoides_df.set_index("cluster_gmm")["n_musicas"]

    # Nosso MVP recebe 10 músicas e recomenda 20
    if contagens.sum() != 10:
        raise ValueError("O input precisa ter 10 músicas.")

    if candidatas_df["track_id"].duplicated().any():
        raise ValueError("Existem track_id duplicados.")

    # Cada música do input dá direito a 2 recomendações
    cotas = contagens * 2

    # Quantas candidatas existem em cada cluster?
    capacidades = (
        candidatas_df.groupby("cluster_gmm")
        .size()
        .reindex(cotas.index, fill_value=0)
    )

    # Impedir cotas maiores que a quantidade disponível
    cotas = pd.Series(
        np.minimum(cotas.to_numpy(), capacidades.to_numpy()),
        index=cotas.index
    )

    # Redistribuir vagas que ficaram sem candidatas
    faltantes = 20 - int(cotas.sum())

    while faltantes > 0:
        houve_redistribuicao = False

        for cluster in contagens.sort_values(ascending=False).index:
            if cotas[cluster] < capacidades[cluster]:
                cotas[cluster] += 1
                faltantes -= 1
                houve_redistribuicao = True

            if faltantes == 0:
                break

        if not houve_redistribuicao:
            raise ValueError("Não existem 20 candidatas suficientes.")

    # Selecionar recomendações dentro de cada cluster
    recomendacoes = []

    for cluster, quantidade in cotas.items():

        if quantidade == 0:
            continue

        grupo = candidatas_df[
            candidatas_df["cluster_gmm"] == cluster
        ].copy()

        # Probabilidades definidas pelo modo escolhido
        probabilidades = calcular_probabilidades(
            grupo["distancia_medoide"].to_numpy(),
            modo=modo
        )

        # Sortear músicas sem repetição
        indices = rng.choice(
            len(grupo),
            size=int(quantidade),
            replace=False,
            p=probabilidades
        )

        recomendacoes.append(grupo.iloc[indices])

    # Juntar as recomendações dos clusters
    playlist = pd.concat(recomendacoes, ignore_index=True)

    # Embaralhar a ordem final
    playlist = playlist.iloc[
        rng.permutation(len(playlist))
    ].reset_index(drop=True)

    return playlist


In [133]:
def calcular_probabilidades(distancias, modo="seguro"):

    # Temperaturas permitidas
    temperaturas = {
        "seguro": 0.3,
        "surpresa": 1.8
    }

    # Validar o modo escolhido
    if modo not in temperaturas:
        raise ValueError(
            "Escolha 'seguro' ou 'surpresa'."
        )

    # Recuperar a temperatura correspondente
    temperature = temperaturas[modo]

    # Converter as distâncias em array NumPy
    distancias = np.asarray(distancias, dtype=float)

    if not np.isfinite(distancias).all():
        raise ValueError("Existem distâncias inválidas.")

    # Normalizar a escala das distâncias
    escala = max(np.std(distancias), 1e-8)

    dist_normalizadas = (
        distancias - distancias.min()
    ) / escala

    # Converter distâncias em pesos
    logits = -dist_normalizadas / temperature
    pesos = np.exp(logits - logits.max())

    # Evitar probabilidades numericamente iguais a zero
    pesos = np.maximum(pesos, 1e-12)

    # Fazer as probabilidades somarem 1
    probabilidades = pesos / pesos.sum()

    return probabilidades

In [135]:
playlist_segura = recomendar_musicas(
    candidatas_com_distancias,
    medoides_df,
    modo="seguro"
)

playlist_surpresa = recomendar_musicas(
    candidatas_com_distancias,
    medoides_df,
    modo="surpresa"
)

for nome, playlist in [
    ("SEGURO", playlist_segura),
    ("SURPRESA", playlist_surpresa)
]:
    print(f"\n=== {nome} ===")
    print("Músicas:", len(playlist))
    print("Distância média:", playlist["distancia_medoide"].mean())

    display(
        playlist[
            ["track_name", "artists", "genres",
             "cluster_gmm", "distancia_medoide"]
        ]
    )

    assert len(playlist) == 20
    assert playlist["track_id"].is_unique
    assert not playlist["track_id"].isin(input_df["track_id"]).any()
    assert playlist["genres"].apply(
        lambda g: set(g).isdisjoint(generos_input)
    ).all()

print("✅ Os dois modos passaram nas validações!")


=== SEGURO ===
Músicas: 20
Distância média: 1.9564283778697038


,track_name,artists,genres,cluster_gmm,distancia_medoide
0,Monkey Wrench,Foo Fighters,[grunge],7,0.630284
1,Good Vibe,Dona Cislene,[hardcore],0,2.557031
2,Maseratiphoïde,Reznyck,[club],4,2.044471
3,Hors-la-loi,Indochine,[synth-pop],0,0.524943
4,Amar y Vivír,Los Pasteles Verdes,[rock-n-roll],0,1.035964
5,Year Zero,Samael,[black-metal],13,0.430049
6,Local God,Panic! At The Disco,[rock],7,1.006319
7,The Exercise Game #2,Mark D. Pencil,[kids],14,6.734942
8,Se Toca Essa Moda,Jads & Jadson,[sertanejo],0,1.438388
9,O Bekhabar,Pritam;Shreya Ghoshal,[indian],11,1.885171



=== SURPRESA ===
Músicas: 20
Distância média: 4.190586053359617


,track_name,artists,genres,cluster_gmm,distancia_medoide
0,Destroyer,Unleash The Archers,[goth],7,2.192437
1,Dejar de ser,A.N.I.M.A.L.;Chuck Jhonson and Juan Jose Burgo...,[heavy-metal],0,6.553351
2,Soldiers Of The Wastelands,DragonForce,[british],4,6.310713
3,My Breath,Dezza;Dan Soleil,[trance],0,4.886511
4,Te Lavaste la Cara y el Mono No,Los Gatos Negros,[punk],0,0.795162
5,The Great Grey Flame,Angelmaker,[black-metal],13,2.039015
6,More Than a Dream / Heart - Live at the O2 Are...,Pet Shop Boys,[synth-pop],7,4.426819
7,シンデレラ・シンドローム,Hysteric Panic,[j-idol],14,10.196414
8,If I Could Turn Back Time,Stephanie J. Block;The Cher Show Ensemble,[show-tunes],0,3.316343
9,Kya,Pritam;Neeraj Shridhar,[indian],11,0.805564


✅ Os dois modos passaram nas validações!
